# Smaug Humanizer — rewrite AI prose on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/smaug-humanizer-llamacpp-1d91/colab/Smaug-Humanizer-llamacpp.ipynb)

[`abacusai/Smaug-Humanizer`](https://huggingface.co/abacusai/Smaug-Humanizer) is Abacus.AI's **~11.96B** rewriter: it takes stiff chatbot prose and returns the same facts, numbers, names, and layout in a more human voice. It is a fine-tune of [`google/gemma-4-12B`](https://huggingface.co/google/gemma-4-12B) (pretrained, not instruction-tuned), trained with SFT on 3,676 rewrite pairs and then 200 steps of GRPO.

Why this model: most "humanizer" demos either paraphrase so hard they drop a date, or they swap two adjectives and still read like a model. Smaug is trained to keep the structure and every number, and to change the rhythm. The card's own advice is to send **about 500 words per call** — split longer drafts at paragraph breaks, rewrite each chunk, and join them with a blank line.

The native BF16 checkpoint is about 24 GB and does not fit a free T4. This notebook loads the **official Q4_K_M GGUF** (~7.38 GB) from [`abacusai/Smaug-Humanizer-GGUF`](https://huggingface.co/abacusai/Smaug-Humanizer-GGUF) through `llama-cpp-python`. **Q5_K_M** (~8.55 GB) is an optional quality bump that still fits ~15 GB with a short context; swap `FILENAME` below if you want it. Do **not** use mradermacher or other third-party re-quants — only the official GGUF repo.

This walkthrough is **inference only**. It does not train or fine-tune anything.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has ~15 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. Do not load BF16 or turn FlashAttention on. This notebook keeps `flash_attn=False` and allocates **8,192** tokens of context, the same window as the GGUF card's `llama-server` example. Weights plus that cache land around 8–10 GB on 15 GB.

**Links:** [model card](https://huggingface.co/abacusai/Smaug-Humanizer) · [GGUF](https://huggingface.co/abacusai/Smaug-Humanizer-GGUF) · [blog](https://blog.abacus.ai/blog/2026/10/26/smaug-humanizer-an-open-weight-12b-humanizer/) · [announcement](https://x.com/bindureddy/status/2114938847704571975)


## 1. Install the CUDA `llama-cpp-python` wheel

Remove any CPU build and install the pinned CUDA 12.1 wheel from the project's GitHub releases. A direct wheel URL stops pip from grabbing the CPU package on PyPI. CUDA 12.1 binaries work with the newer driver on a Colab T4. Current free Colab images (Python 3.13) ship the NVIDIA driver but not the CUDA 12 user-space libraries the wheel links (`libcudart.so.12`, `libcublas.so.12`), so the install also pulls `nvidia-cuda-runtime-cu12` and `nvidia-cublas-cu12`.

**0.3.36** (1 October 2026) is the current CUDA 12.1 wheel. Gemma 4 chat support landed in llama-cpp-python 0.3.25; the official Smaug GGUF was built with llama.cpp `b11899` and needs a build that already has Gemma 4. Use `%pip` (not `python -I -m pip`) so Colab's site-packages stay on the path.


In [ ]:
%pip uninstall -qqq -y llama-cpp-python
# Free Colab T4 (Python 3.13) has a driver but no CUDA 12 user-space libs.
# The cu121 wheel dynamically links libcudart.so.12 and libcublas.so.12.
%pip install -qqq --no-cache-dir nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
%pip install -qqq --no-cache-dir https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.36-cu121/llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl

# Direct wheel URL, same pattern as colab/LFM-2_5-2_6b-llamacpp-python.ipynb
# and colab/Mellum2.1-12B-A2.5B-Thinking-coding-agent.ipynb.
# 0.3.36 is after llama.cpp gained Gemma 4 (needed for this GGUF).


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Native libraries already imported cannot be replaced in the same process. If you skip this after the install, the next cell stops with a version check.


## 2. Check the GPU and print T4 caveats


In [ ]:
import ctypes
import glob
import importlib
import os
import subprocess

import torch
from packaging.version import Version


def _preload_cuda12_runtime() -> None:
    """Make CUDA 12 runtime/cublas visible before importing llama_cpp.

    The cu121 wheel CDLLs libggml-cuda, which needs libcudart.so.12 and
    libcublas.so.12. Those live in the nvidia-*-cu12 pip packages on current
    free Colab, not on the default linker path.
    """
    sos = []
    for pkg_name in ("nvidia.cuda_runtime", "nvidia.nvjitlink", "nvidia.cublas"):
        try:
            pkg = importlib.import_module(pkg_name)
        except ImportError:
            continue
        sos.extend(glob.glob(os.path.join(pkg.__path__[0], "lib", "*.so*")))
    # Two passes so cublas can resolve cublasLt / nvJitLink loaded by full path.
    for so in sos + sos:
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            continue


def print_vram(label=""):
    # llama.cpp allocates CUDA memory outside PyTorch. nvidia-smi is the measured counter.
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
    suffix = f" ({label})" if label else ""
    print(f"nvidia-smi used/free/total{suffix}: {smi}")


_preload_cuda12_runtime()
import llama_cpp

assert Version(llama_cpp.__version__) >= Version("0.3.36"), (
    f"llama-cpp-python {llama_cpp.__version__} is loaded; Smaug's Gemma 4 GGUF needs 0.3.36+. "
    "Run the install cell, then Runtime > Restart session."
)

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

!nvidia-smi

major, minor = torch.cuda.get_device_capability()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
name = torch.cuda.get_device_name()

print()
print("=== T4-first caveats ===")
print(f"GPU: {name}  sm_{major}{minor}  VRAM: {vram_gb:.1f} GB")
print(f"llama-cpp-python {llama_cpp.__version__}")
print("- No native bfloat16 on T4 (sm_75). This notebook loads official Q4_K_M GGUF, not BF16.")
print("- Native BF16 weights are ~24 GB and will OOM a 16 GB card.")
print("- No FlashAttention-2 on sm_75. flash_attn=False (sdpa / eager equivalent).")
print("- Official Q4_K_M GGUF is ~7.38 GB; 8K context still fits on 15 GB.")
print("- Official Q5_K_M (~8.55 GB) is optional if you want a larger quant on the same T4.")
print("- Do not download mradermacher or other third-party re-quants.")
print("- Card guidance: about 500 words per call. Split longer drafts at paragraph breaks.")
print_vram("before load")
if major < 8:
    print("- FlashAttention stays off on this GPU.")


## 3. Load the official Q4_K_M GGUF

`Llama.from_pretrained` downloads `Smaug-Humanizer-Q4_K_M.gguf` (~7.38 GB) from [`abacusai/Smaug-Humanizer-GGUF`](https://huggingface.co/abacusai/Smaug-Humanizer-GGUF). `n_gpu_layers=-1` offloads every layer to the T4. `n_ctx=8192` matches the GGUF card. `flash_attn=False` is required on sm_75.

To try **Q5_K_M** (~8.55 GB) instead, change `FILENAME` to `Smaug-Humanizer-Q5_K_M.gguf`. Leave `Q6_K` / `Q8_0` / `BF16` alone on a free T4.

Sampler defaults match the [model card](https://huggingface.co/abacusai/Smaug-Humanizer): temperature 1.0, top_p 0.95, top_k 64. The model ends a turn with the literal string `<end_of_turn>` (not `<eos>`), so that is a stop string.


In [ ]:
import os

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

from llama_cpp import Llama

REPO_ID = "abacusai/Smaug-Humanizer-GGUF"
FILENAME = "Smaug-Humanizer-Q4_K_M.gguf"
# Official Q5_K_M (~8.55 GB) if you want a larger quant on the same T4:
# FILENAME = "Smaug-Humanizer-Q5_K_M.gguf"
N_CTX = 8192

# Card sampler. Keep these next to the Llama so later cells reuse them.
SAMPLER = dict(temperature=1.0, top_p=0.95, top_k=64)
STOP = ["<end_of_turn>"]

llm = Llama.from_pretrained(
    repo_id=REPO_ID,
    filename=FILENAME,
    n_gpu_layers=-1,
    n_ctx=N_CTX,
    flash_attn=False,
    verbose=True,
)

print()
print("loaded", FILENAME, "| n_ctx", N_CTX, "| n_gpu_layers -1 | flash_attn False")
print("sampler", SAMPLER, "| stop", STOP)
print()
print_vram("after load")
print()
print("GPU status after load:")
!nvidia-smi


## 4. Humanize helper

The prompt is the one on the model card: one user turn, rewrite only, keep facts and layout. `chunk_text` splits on paragraph breaks when a draft is longer than about **500 words**, which is the card's recommended call size. The three demos below are shorter than that on purpose so a free T4 can finish them in one session.


In [ ]:
PROMPT = """Rewrite the AI-written text below so that it reads as if a person wrote it, in a person's own voice, wording and rhythm.

Keep every fact, number, name and quote. Keep the structure and the language of the text. Output only the rewritten text.

Text:
{text}"""

CHUNK_WORDS = 500


def word_count(text: str) -> int:
    return len(text.split())


def chunk_text(text: str, max_words: int = CHUNK_WORDS) -> list[str]:
    """Split on blank lines so each piece stays near the card's 500-word budget."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current, current_words = [], [], 0
    for para in paragraphs:
        n = word_count(para)
        if current and current_words + n > max_words:
            chunks.append("\n\n".join(current))
            current, current_words = [para], n
        else:
            current.append(para)
            current_words += n
    if current:
        chunks.append("\n\n".join(current))
    return chunks


def humanize(text: str, max_tokens: int = 1024) -> str:
    """Rewrite one passage. Long drafts are chunked at ~500 words, then joined."""
    pieces = []
    for i, chunk in enumerate(chunk_text(text), start=1):
        n = word_count(chunk)
        print(f"--- chunk {i}: {n} words (card budget ~{CHUNK_WORDS}) ---")
        out = llm.create_chat_completion(
            messages=[{"role": "user", "content": PROMPT.format(text=chunk)}],
            max_tokens=max_tokens,
            stop=STOP,
            **SAMPLER,
        )
        rewrite = (out["choices"][0]["message"].get("content") or "").strip()
        for token in STOP:
            if rewrite.endswith(token):
                rewrite = rewrite[: -len(token)].strip()
        pieces.append(rewrite)
        usage = out.get("usage", {})
        print(
            f"prompt {usage.get('prompt_tokens', '?')} | "
            f"completion {usage.get('completion_tokens', '?')} | "
            f"finish {out['choices'][0].get('finish_reason')}"
        )
        print_vram(f"after chunk {i}")
    return "\n\n".join(pieces)


print("chunk helper ready | max words per call", CHUNK_WORDS)


## 5. Demo 1 — stiff product update

A changelog-shaped announcement with the usual AI filler ("leverage", "seamless", "robust"). Facts that must survive: 14 May 2026, 12% p95 latency drop, 3 regions, 2,400 tenants, 99.9% uptime, the 48-hour webhook retry.


In [ ]:
DEMO_1 = """We are pleased to announce that, effective 14 May 2026, Northwind Billing will leverage a seamless, end-to-end invoicing experience designed to empower finance teams at scale.

This robust enhancement reduces p95 invoice-generation latency by 12%, from 840 ms to 739 ms, across our 3 regions (us-east-1, eu-west-1, and ap-southeast-1). More than 2,400 tenants already process payroll on the prior stack, and the cutover preserves the existing 99.9% monthly uptime objective.

In addition, failed webhook deliveries will now automatically retry for 48 hours with exponential backoff, ensuring stakeholders remain informed throughout the payment lifecycle. We remain committed to delivering best-in-class, holistic value for our customers."""

print("=== input ===")
print(DEMO_1)
print(f"\n({word_count(DEMO_1)} words)\n")
print("=== rewrite ===")
print(humanize(DEMO_1))


## 6. Demo 2 — generic explainer

A moon-phase explainer in the same over-smooth register the card uses as an example. Facts that must survive: 29.5 days, new / first quarter / full / last quarter, sunlight on the near side, the Earth–Moon–Sun angle.


In [ ]:
DEMO_2 = """The phases of the Moon constitute a fundamental astronomical phenomenon that occurs as a result of the changing geometric relationship between the Earth, the Moon, and the Sun.

Over a period of approximately 29.5 days, observers on Earth perceive a systematic sequence of illumination: new Moon, first quarter, full Moon, and last quarter. It is important to note that the Moon does not emit its own light; rather, the visible portion is sunlight reflected from the lunar near side.

A comprehensive understanding of this cycle enables learners to appreciate that the phase they see is determined by the Earth–Moon–Sun angle, not by the Earth's shadow, which is responsible for lunar eclipses instead. This distinction is essential for avoiding a common misconception."""

print("=== input ===")
print(DEMO_2)
print(f"\n({word_count(DEMO_2)} words)\n")
print("=== rewrite ===")
print(humanize(DEMO_2))


## 7. Demo 3 — support email that sounds like a template

A "thank you for reaching out" reply. Facts that must survive: ticket #48117, 3 April, $42.00, order W-2291, refund in 5–7 business days, reply within 24 hours, Maya in billing support.


In [ ]:
DEMO_3 = """Thank you for reaching out to Northwind Support regarding ticket #48117, which you opened on 3 April. We sincerely appreciate the opportunity to assist you.

We have carefully reviewed your request and can confirm that the $42.00 charge on order W-2291 was a duplicate capture. A refund has been initiated and should appear on the original payment method in 5–7 business days.

Please do not hesitate to reply to this thread if you have any further questions. We are here to help and will respond within 24 hours. Kind regards,
Maya
Billing Support"""

print("=== input ===")
print(DEMO_3)
print(f"\n({word_count(DEMO_3)} words)\n")
print("=== rewrite ===")
print(humanize(DEMO_3))


## 8. Caveats and cleanup

**What this notebook is for.** Inference-only humanizing on a free T4 with the official Q4_K_M GGUF and the card's prompt / sampler / stop string. The three demos are short on purpose; `chunk_text` is there for drafts that go past ~500 words.

**What it is not.** Training, detector evasion as a product, or a guarantee that every rewrite will pass a commercial AI-text detector. The card is explicit: Pangram still flags some outputs, more often on long texts. Sample again if it matters, and read the rewrite before you publish it.

**Chunk size.** Split long inputs at paragraph breaks into chunks of about **500 words**, rewrite each chunk, and join them with a blank line. Detectors score a long document in windows; one leftover AI paragraph can flag the whole thing.

**T4 notes.** Official Q4_K_M (~7.38 GB) plus `n_ctx=8192` is the free-T4 budget. Q5_K_M (~8.55 GB) is the next official quant if you want more quality on the same 15 GB card. Do not load BF16, and do not use third-party re-quants. `print_vram` reports `nvidia-smi` used/free/total because llama.cpp does not go through `torch.cuda.memory_allocated`.

**Stop string.** Add `<end_of_turn>`. In the GGUF that text is not a special token, so clients that omit it will print it at the end of the answer.

**Licence.** Apache 2.0, same as [`google/gemma-4-12B`](https://huggingface.co/google/gemma-4-12B).

**Links again:** [model card](https://huggingface.co/abacusai/Smaug-Humanizer) · [GGUF](https://huggingface.co/abacusai/Smaug-Humanizer-GGUF) · [blog](https://blog.abacus.ai/blog/2026/10/26/smaug-humanizer-an-open-weight-12b-humanizer/) · [announcement](https://x.com/bindureddy/status/2114938847704571975)

Free the GPU before you open another notebook:


In [ ]:
import gc

print_vram("before cleanup")
del llm
gc.collect()
print_vram("after cleanup")
